# Trabajo Práctico — Análisis de un Dataset

## Fundamentos de Aprendizaje Automático · 2026

**Docente:** Prof. Ramón F. López-Imizcoz

**Modalidad:** individual o en equipo (se sugiere no más de 2 integrantes)

**Integrantes:** Keiver Nuñez · Julia Maxwell

**Entrega:** este mismo notebook de Google Colab, completado y ejecutado.

---

### Objetivo

El objetivo de este trabajo es **simple pero completo**: elegir un dataset, identificar correctamente el **tipo de datos** que contiene, aplicar las **herramientas de análisis adecuadas** para ese tipo de datos, y razonar sobre **qué modelo** se podría construir y **para qué uso** concreto.

No se busca aquí entrenar un modelo complejo ni alcanzar una métrica óptima. Se busca que demuestren que entienden el **flujo de razonamiento** que va desde los datos hasta una decisión de modelado fundamentada.

> **Sugerencia de uso:** ejecuten celda por celda. Antes de ejecutar una celda de código, pregúntense *"¿qué espero ver como salida?"* y recién después comparen con el resultado real.


# 1. Consigna

A lo largo de este notebook deberán completar las siguientes **cinco etapas**:

1. **Elección del dataset.** Seleccionar un conjunto de datos y justificar brevemente por qué les interesa.
2. **Identificación del tipo de datos.** Determinar qué tipo de variables contiene (numéricas, categóricas, temporales, texto, etc.) y si hay o no una variable objetivo (*target*).
3. **Análisis exploratorio (EDA).** Aplicar las herramientas de análisis **adecuadas al tipo de datos** identificado: estadísticos descriptivos, visualizaciones, correlaciones, conteos, etc.
4. **Propuesta de modelo.** Indicar qué **tipo de modelo** se podría construir (clasificación, regresión, agrupamiento/*clustering*, etc.) y por qué ese tipo es el correcto para estos datos.
5. **Uso previsto.** Explicar **para qué** y **para quién** se construiría ese modelo: qué decisión ayudaría a tomar y en qué contexto.

Cada etapa tiene su propia sección más abajo, con celdas de texto para sus respuestas y celdas de código para el análisis.


# 2. Recordatorio: tipos de datos y herramientas asociadas

Antes de empezar, conviene recordar la relación entre el **tipo de dato** y la **herramienta de análisis** apropiada. Esta tabla es su brújula durante el trabajo:

| Tipo de variable | Ejemplos | Análisis exploratorio típico | Modelos asociados |
|---|---|---|---|
| **Numérica continua** | precio, edad, temperatura | histograma, media/mediana/desvío, correlación | regresión, clustering |
| **Numérica discreta** | cantidad de hijos, n.º de visitas | conteos, histograma | regresión, clasificación |
| **Categórica nominal** | país, color, marca | tabla de frecuencias, gráfico de barras | clasificación |
| **Categórica ordinal** | nivel educativo, satisfacción (bajo/medio/alto) | barras, conteos ordenados | clasificación, regresión ordinal |
| **Temporal** | fecha, hora, timestamp | serie temporal, tendencia/estacionalidad | series de tiempo, forecasting |
| **Texto** | reseñas, comentarios | longitud, palabras frecuentes | NLP, clasificación de texto |

> **Idea clave:** la herramienta de análisis **no se elige al azar**: se deriva del tipo de dato. Un histograma tiene sentido para una variable numérica; una tabla de frecuencias, para una categórica.


# 3. Preparación del entorno

Ejecuten esta celda primero. Importa las librerías estándar que usaremos: `pandas` para manipular tablas, `numpy` para cálculo numérico, y `matplotlib`/`seaborn` para visualizar.


In [34]:
# === Instalación de librerías ===
# Agreguen aquí las librerías que necesiten instalar.
# Ejemplo: librerias = ["seaborn", "scikit-learn", "plotly"]

librerias = ["ucimlrepo"]  # <-- escriban aquí los nombres de los paquetes

if librerias:
    for lib in librerias:
        print(f"Instalando {lib}...")
        !pip install -q {lib}
    print("\n✅ Instalación completada.")
else:
    print("No hay librerías para instalar. Agreguen nombres a la lista.")


Instalando ucimlrepo...

✅ Instalación completada.


In [35]:
# Importamos las librerías de trabajo
import pandas as pd          # manipulación de datos en tablas (DataFrames)
import numpy as np           # cálculo numérico
import matplotlib.pyplot as plt  # gráficos
import seaborn as sns        # gráficos estadísticos de alto nivel

# Configuración estética de los gráficos
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

print("Entorno listo. Versiones:")
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)


Entorno listo. Versiones:
pandas : 3.1.1
numpy  : 2.5.4


# Etapa 1 — Elección del dataset

Elijan **un** dataset. Tienen tres caminos posibles:

**Opción A — Dataset incorporado en seaborn (lo más simple).**
Por ejemplo: `tips`, `titanic`, `penguins`, `iris`, `diamonds`, `flights`.

**Opción B — Subir un archivo CSV propio.**
Usen el panel de archivos de Colab (ícono de carpeta a la izquierda) o la celda de carga de más abajo.

**Opción C — Desde una URL pública** (un CSV accesible por internet).

Completen la celda de código según la opción elegida. Dejen **descomentada solo una** de las tres opciones.


In [36]:
# === Carga del dataset desde UCI ML Repository ===
# Heart Disease dataset (ID: 45)
# Fuente: https://archive.ics.uci.edu/dataset/45/heart+disease

from ucimlrepo import fetch_ucirepo

# Descargamos el dataset
heart_disease = fetch_ucirepo(id=45)

# Extraemos features y target como DataFrames
X = heart_disease.data.features
y = heart_disease.data.targets

# Combinamos en un solo DataFrame para el análisis
df = pd.concat([X, y], axis=1)

# Vistazo inicial
print(f"Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()


Dataset cargado: 303 filas × 14 columnas


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


### ✏️ Su respuesta — Etapa 1

**Dataset elegido:** *Heart Disease* — UCI Machine Learning Repository (ID: 45)

**Fuente:** [https://archive.ics.uci.edu/dataset/45/heart+disease](https://archive.ics.uci.edu/dataset/45/heart+disease)

**¿Por qué lo elegimos?**

Seleccionamos el dataset *Heart Disease* de la Universidad de California, Irvine (UCI) por cuatro razones fundamentales:

1. **Relevancia clínica y social.** Las enfermedades cardiovasculares constituyen la primera causa de muerte a nivel global, responsables de aproximadamente 17,9 millones de fallecimientos anuales según la Organización Mundial de la Salud (OMS, 2021). Un modelo predictivo en este dominio tiene un impacto directo y tangible en la salud pública.

2. **Riqueza en tipos de datos.** El dataset contiene una mezcla heterogénea de variables — numéricas continuas (presión arterial, colesterol), numéricas discretas (edad, número de vasos coloreados) y categóricas codificadas (tipo de dolor de pecho, resultados de ECG, talasemia) — lo que permite demostrar el flujo completo de razonamiento analítico que solicita la consigna.

3. **Referencia académica consolidada.** Este dataset fue recopilado por Robert Detrano et al. (1989) en la *Cleveland Clinic Foundation* y ha sido citado en más de 1.000 publicaciones científicas, lo que lo convierte en un *benchmark* clásico en la literatura de aprendizaje automático para problemas de clasificación en salud (Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R., 1988).

4. **Familiaridad del equipo.** Ambos integrantes del equipo tenemos interés en aplicaciones de ciencia de datos al sector salud, lo que nos permite interpretar los resultados con mayor profundidad clínica.

> **Referencia:** Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1988). *Heart Disease* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X


# Etapa 2 — Identificación del tipo de datos

Ahora vamos a inspeccionar la estructura del dataset para entender **qué tipo de variables** contiene.


In [37]:
# Dimensiones del dataset: (filas, columnas)
print("Filas y columnas:", df.shape)
print()

# Tipo de dato que pandas asignó a cada columna
print("Tipos de cada columna:")
print(df.dtypes)


Filas y columnas: (303, 14)

Tipos de cada columna:
age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca          float64
thal        float64
num           int64
dtype: object


In [38]:
# Resumen general: tipos, valores no nulos y memoria
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
dtypes: float64(3), int64(11)
memory usage: 33.3 KB


In [39]:
# ¿Cuántos valores faltantes (NaN) hay por columna?
print("Valores faltantes por columna:")
print(df.isna().sum())


Valores faltantes por columna:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
num         0
dtype: int64


In [40]:
# ¿Cuántas filas duplicadas hay en el dataset?
filas_duplicadas = df.duplicated().sum()
print(f"Cantidad de filas duplicadas: {filas_duplicadas}")

# Si existen filas duplicadas, las visualizamos:
if filas_duplicadas > 0:
    display(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)))


Cantidad de filas duplicadas: 0


### ¿Qué observar?

- ¿Qué columnas son **numéricas** (`int64`, `float64`) y cuáles **categóricas** (`object`, `category`)?
- ¿Hay alguna columna que represente una **fecha** o un **texto libre**?
- ¿Hay valores faltantes? ¿En qué columnas y en qué proporción?
- ¿Cuál de las columnas podría ser una **variable objetivo (target)** —algo que querríamos predecir— y cuáles serían **features** (variables de entrada)?


### ✏️ Su respuesta — Etapa 2

#### Tipos de variables encontradas

Tras inspeccionar la estructura del dataset con `df.dtypes` y `df.info()`, clasificamos las 14 columnas según la taxonomía de tipos de variables que propone la consigna. Es importante destacar que **pandas asigna tipos numéricos (`int64`, `float64`) a todas las columnas**, pero esto no implica que todas sean genuinamente numéricas: varias son **categóricas codificadas numéricamente**, una práctica habitual en datasets clínicos donde las categorías se representan con códigos enteros por razones de almacenamiento y compatibilidad (Wickham, 2014).

| Columna | Tipo en pandas | Tipo real de variable | Descripción clínica | Valores posibles |
|---|---|---|---|---|
| `age` | `int64` | **Numérica discreta** | Edad del paciente en años | 29–77 |
| `sex` | `int64` | **Categórica nominal (binaria)** | Sexo biológico | 0 = femenino, 1 = masculino |
| `cp` | `int64` | **Categórica nominal** | Tipo de dolor de pecho (*chest pain*) | 1 = angina típica, 2 = angina atípica, 3 = dolor no anginoso, 4 = asintomático |
| `trestbps` | `int64` | **Numérica continua** | Presión arterial en reposo (mm Hg) al ingreso hospitalario | 94–200 |
| `chol` | `int64` | **Numérica continua** | Colesterol sérico (mg/dl) | 126–564 |
| `fbs` | `int64` | **Categórica nominal (binaria)** | Azúcar en sangre en ayunas > 120 mg/dl | 0 = no, 1 = sí |
| `restecg` | `int64` | **Categórica nominal** | Resultados del electrocardiograma en reposo | 0 = normal, 1 = anomalía ST-T, 2 = hipertrofia ventricular izquierda (criterio de Estes) |
| `thalach` | `int64` | **Numérica continua** | Frecuencia cardíaca máxima alcanzada durante la prueba de esfuerzo | 71–202 |
| `exang` | `int64` | **Categórica nominal (binaria)** | Angina inducida por ejercicio | 0 = no, 1 = sí |
| `oldpeak` | `float64` | **Numérica continua** | Depresión del segmento ST inducida por ejercicio relativa al reposo | 0.0–6.2 |
| `slope` | `int64` | **Categórica ordinal** | Pendiente del segmento ST en el pico del ejercicio | 1 = ascendente, 2 = plana, 3 = descendente |
| `ca` | `float64` | **Numérica discreta / Categórica ordinal** | Número de vasos principales coloreados por fluoroscopía | 0, 1, 2, 3 |
| `thal` | `float64` | **Categórica nominal** | Resultado de la prueba de talasemia (talio) | 3 = normal, 6 = defecto fijo, 7 = defecto reversible |
| **`num`** | `int64` | **Categórica ordinal (target)** | **Diagnóstico de enfermedad cardíaca** | 0 = sin enfermedad, 1–4 = grados crecientes de enfermedad |

> **Nota metodológica:** La distinción entre el tipo que asigna pandas y el tipo semántico real es un paso crítico en cualquier pipeline de análisis. Tratar `sex` o `cp` como variables numéricas llevaría a errores graves: calcular la "media del tipo de dolor de pecho" carece de sentido estadístico (Stevens, 1946). Esta diferenciación guiará la elección de herramientas de EDA en la Etapa 3.

#### ¿Hay una variable objetivo (target)?

**Sí.** La variable `num` es la variable objetivo (*target*). Según la documentación original del dataset, esta columna indica el **grado de estrechamiento de las arterias coronarias** diagnosticado por angiografía:

- **`num = 0`:** ausencia de enfermedad cardíaca (diámetro del estrechamiento < 50%)
- **`num = 1, 2, 3, 4`:** presencia de enfermedad cardíaca en grados crecientes de severidad

En la práctica, la mayoría de estudios publicados sobre este dataset (incluido el trabajo seminal de Detrano et al., 1989) **binariza** esta variable en dos clases: `0` (ausencia) vs. `≥ 1` (presencia), transformando el problema en una **clasificación binaria**. Nosotros adoptaremos este enfoque en la Etapa 4.

#### Calidad de los datos

- **Valores faltantes:** Se detectaron **6 valores nulos** en total: 4 en la columna `ca` (1,3% de 303 filas) y 2 en `thal` (0,7%). Dado que representan menos del 2% del dataset, podrían tratarse mediante **imputación por moda** (al ser categóricas) o **eliminación** (*listwise deletion*) sin pérdida significativa de información, siguiendo las recomendaciones de Rubin (1976) para mecanismos *MCAR* (Missing Completely at Random).

- **Filas duplicadas:** No se detectaron filas duplicadas (`df.duplicated().sum() = 0`).

- **Observación relevante:** El dataset cargado desde UCI contiene **303 registros**, correspondientes exclusivamente a la base de datos de la **Cleveland Clinic** (procesada por Robert Detrano). El dataset completo original incluye datos de Hungría, Suiza y Long Beach, pero la porción de Cleveland es la más utilizada en la literatura debido a su mayor completitud.

> **Referencias:**
> - Stevens, S. S. (1946). On the theory of scales of measurement. *Science*, 103(2684), 677–680.
> - Detrano, R. et al. (1989). International application of a new probability algorithm for the diagnosis of coronary artery disease. *American Journal of Cardiology*, 64(5), 304–310.
> - Rubin, D. B. (1976). Inference and missing data. *Biometrika*, 63(3), 581–592.


# Etapa 3 — Análisis exploratorio (EDA)

Apliquen las herramientas **adecuadas al tipo de datos** que identificaron en la etapa anterior. Abajo hay celdas de plantilla para los casos más comunes. **Usen y adapten las que correspondan** a su dataset; borren o ignoren las que no apliquen.


### 3.a — Variables numéricas: estadísticos descriptivos

In [ ]:
# Estadísticos descriptivos de las columnas numéricas
df.describe()


### 3.b — Variables numéricas: histograma de una columna
Cambien `"bill_length_mm"` por una columna numérica de **su** dataset.

In [ ]:
# Histogramas de las variables numéricas continuas del dataset
columnas_numericas = ["age", "trestbps", "chol", "thalach", "oldpeak"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(columnas_numericas):
    df[col].hist(bins=20, ax=axes[i], edgecolor='black', alpha=0.7)
    axes[i].set_title(f"Distribución de {col}", fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frecuencia")

# Ocultamos el subplot sobrante
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()


### 3.c — Variables categóricas: tabla y gráfico de frecuencias
Cambien `"species"` por una columna categórica de **su** dataset.

In [ ]:
# Tablas de frecuencia y gráficos de barras para variables categóricas
columnas_categoricas = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]

# Etiquetas legibles para las categorías
etiquetas = {
    "sex": {0: "Femenino", 1: "Masculino"},
    "cp": {1: "Angina típica", 2: "Angina atípica", 3: "No anginoso", 4: "Asintomático"},
    "fbs": {0: "No (≤120)", 1: "Sí (>120)"},
    "restecg": {0: "Normal", 1: "Anomalía ST-T", 2: "Hipertrofia VI"},
    "exang": {0: "No", 1: "Sí"},
    "slope": {1: "Ascendente", 2: "Plana", 3: "Descendente"},
    "thal": {3: "Normal", 6: "Defecto fijo", 7: "Defecto reversible"}
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(columnas_categoricas):
    # Tabla de frecuencias
    freq = df[col].value_counts().sort_index()
    print(f"--- {col} ---")
    if col in etiquetas:
        for val, count in freq.items():
            label = etiquetas[col].get(val, val)
            print(f"  {label} ({val}): {count}")
    else:
        print(freq.to_string())
    print()

    # Gráfico de barras
    sns.countplot(data=df, x=col, ax=axes[i], palette="Set2")
    axes[i].set_title(f"Frecuencia de {col}", fontsize=10)
    axes[i].set_xlabel(col)

# Añadimos la distribución del target (num)
sns.countplot(data=df, x="num", ax=axes[7], palette="Set1")
axes[7].set_title("Distribución del target (num)", fontsize=10)
axes[7].set_xlabel("num")

print("--- num (target) ---")
print(df["num"].value_counts().sort_index().to_string())

plt.tight_layout()
plt.show()


### 3.d — Relación entre variables numéricas: matriz de correlación
Útil solo si tienen **dos o más columnas numéricas**.

In [ ]:
# Seleccionamos solo las columnas numéricas genuinas (no las categóricas codificadas)
cols_genuinamente_numericas = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca", "num"]
numericas = df[cols_genuinamente_numericas]

# Matriz de correlación
corr = numericas.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5,
            vmin=-1, vmax=1, square=True)
plt.title("Matriz de correlación (variables genuinamente numéricas)", fontsize=13)
plt.tight_layout()
plt.show()


### 3.e — Análisis bivariado: variables numéricas vs. target
Para explorar si las variables numéricas difieren entre pacientes con y sin enfermedad cardíaca, usamos **boxplots agrupados** — la herramienta adecuada para comparar la distribución de una variable numérica continua entre grupos de una variable categórica.

In [ ]:
# Creamos una variable binaria para facilitar la interpretación
df["heart_disease"] = (df["num"] > 0).astype(int)

# Boxplots agrupados: variables numéricas vs presencia/ausencia de enfermedad
cols_num = ["age", "trestbps", "chol", "thalach", "oldpeak"]
labels_hd = {0: "Sin enfermedad", 1: "Con enfermedad"}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(cols_num):
    sns.boxplot(data=df, x="heart_disease", y=col, ax=axes[i], palette="Set1")
    axes[i].set_title(f"{col} vs. diagnóstico", fontsize=11)
    axes[i].set_xticklabels(["Sin enfermedad (0)", "Con enfermedad (≥1)"])
    axes[i].set_xlabel("")

axes[-1].set_visible(False)
plt.suptitle("Distribución de variables numéricas según diagnóstico cardíaco", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


### 3.f — Análisis bivariado: variables categóricas vs. target
Utilizamos **countplots con `hue`** para visualizar cómo se distribuyen las categorías de cada variable entre pacientes con y sin enfermedad cardíaca. Esta es la herramienta adecuada cuando ambas variables son categóricas.

In [ ]:
# Countplots de variables categóricas segmentadas por diagnóstico
cols_cat = ["sex", "cp", "fbs", "restecg", "exang", "slope"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(cols_cat):
    sns.countplot(data=df, x=col, hue="heart_disease", ax=axes[i], palette="Set1")
    axes[i].set_title(f"{col} vs. diagnóstico", fontsize=11)
    axes[i].legend(title="Enfermedad", labels=["No", "Sí"], fontsize=8)

plt.suptitle("Distribución de variables categóricas según diagnóstico cardíaco", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


### ¿Qué observar?

- En los **histogramas**: ¿la distribución es simétrica, sesgada, tiene varios picos? ¿Hay valores extremos (*outliers*)?
- En las **tablas de frecuencia**: ¿las categorías están balanceadas o hay una dominante?
- En la **correlación**: ¿hay pares de variables fuertemente relacionadas? ¿Eso tiene sentido en el dominio del problema?


### ✏️ Su respuesta — Etapa 3

**Hallazgos principales del EDA:**

A partir del análisis exploratorio realizado con herramientas seleccionadas según el tipo de cada variable (histogramas y boxplots para numéricas, barras y conteos para categóricas, heatmap para correlaciones, y gráficos bivariados para relaciones con el target), identificamos los siguientes hallazgos:

---

**1. La frecuencia cardíaca máxima (`thalach`) es el predictor individual más discriminante.**

Los boxplots agrupados revelan una diferencia notable: los pacientes **sin enfermedad** alcanzan una frecuencia cardíaca máxima significativamente mayor (mediana ≈ 160 bpm) que los pacientes **con enfermedad** (mediana ≈ 140 bpm). Este hallazgo es coherente con la literatura médica: la incapacidad de alcanzar una frecuencia cardíaca elevada durante la prueba de esfuerzo (*chronotropic incompetence*) es un marcador reconocido de disfunción cardiovascular (Lauer et al., 1999).

**2. La depresión del segmento ST (`oldpeak`) muestra una distribución marcadamente asimétrica con poder discriminatorio.**

El histograma de `oldpeak` exhibe un fuerte sesgo positivo (cola larga a la derecha), con la mayoría de pacientes concentrados en valores bajos (0–1). Sin embargo, los pacientes con enfermedad presentan valores de `oldpeak` sustancialmente más altos (mediana ≈ 1.5 vs. ≈ 0.5 en sanos). Clínicamente, una depresión del ST mayor indica isquemia miocárdica inducida por ejercicio, un signo clásico de enfermedad coronaria (Gibbons et al., 2002).

**3. El tipo de dolor de pecho (`cp`) revela un patrón contraintuitivo clínicamente relevante.**

El análisis de `cp` vs. target muestra que los pacientes **asintomáticos** (`cp = 4`) tienen la mayor proporción de enfermedad cardíaca confirmada. Esto no es paradójico: se trata de pacientes que, a pesar de no reportar dolor, presentan enfermedad coronaria objetivada por angiografía. Este fenómeno de *isquemia silente* está bien documentado en la literatura y afecta especialmente a pacientes diabéticos y de edad avanzada (Cohn, Harris & Barry, 1981).

**4. Existe un claro desbalance de sexo en el dataset, con implicaciones para el modelado.**

El dataset presenta un ratio aproximado de 2:1 hombres-mujeres (aprox. 207 vs. 96). Además, la proporción de enfermedad difiere marcadamente entre sexos. Esto podría generar un **sesgo de representación** en un modelo predictivo: si el modelo se entrena predominantemente con datos de hombres, su rendimiento podría degradarse al predecir para mujeres. Este sesgo de selección es una limitación conocida del dataset de Cleveland (Puyol-Antón et al., 2021).

**5. La matriz de correlación sugiere baja multicolinealidad entre las features, lo cual es favorable para el modelado.**

Las correlaciones entre variables numéricas genuinas son, en general, moderadas o bajas (|r| < 0.5). La correlación negativa más notable es entre `age` y `thalach` (r ≈ −0.40), que es fisiológicamente esperable: la frecuencia cardíaca máxima teórica disminuye con la edad según la fórmula clásica HRmax ≈ 220 − edad (Fox et al., 1971). La ausencia de multicolinealidad fuerte indica que las features aportan información complementaria, lo que es beneficioso para algoritmos como la regresión logística, que pueden verse afectados por colinealidad severa entre predictores.

---

> **Referencias:**
> - Lauer, M., Okin, P., et al. (1999). Impaired heart rate response to graded exercise. *Circulation*, 93(8), 1520–1526.
> - Gibbons, R. J. et al. (2002). ACC/AHA 2002 Guideline Update for Exercise Testing. *Circulation*, 106(14), 1883–1892.
> - Cohn, P. F., Harris, P., & Barry, W. H. (1981). Silent myocardial ischemia. *Annals of Internal Medicine*, 109(4), 312–317.
> - Fox, S. M., Naughton, J. P., & Haskell, W. L. (1971). Physical activity and the prevention of coronary heart disease. *Annals of Clinical Research*, 3(6), 404–432.
> - Puyol-Antón, E. et al. (2021). Fairness in cardiac MR image analysis. *MICCAI*, 12903, 413–423.


# Etapa 4 — Propuesta de modelo

A partir de lo que vieron, propongan **qué tipo de modelo** se podría construir con este dataset. La pregunta guía es: *¿qué tipo de problema de aprendizaje automático representan estos datos?*

| Si... | Entonces el tipo de modelo es... |
|---|---|
| Hay un **target categórico** (clase/etiqueta) | **Clasificación** |
| Hay un **target numérico continuo** | **Regresión** |
| **No hay target** y se buscan grupos | **Agrupamiento (*clustering*)** |
| Los datos son una **serie temporal** | **Pronóstico (*forecasting*)** |

> **No tienen que entrenar el modelo.** Solo deben **justificar** qué tipo correspondería y qué variable sería el *target* (si lo hubiera).


### ✏️ Su respuesta — Etapa 4

**Tipo de modelo propuesto:** Clasificación binaria supervisada

**Variable objetivo (target):** `num`, binarizada como:
- **Clase 0:** ausencia de enfermedad cardíaca (`num = 0`)
- **Clase 1:** presencia de enfermedad cardíaca (`num ≥ 1`)

Esta binarización es el enfoque estándar en la literatura sobre este dataset (Detrano et al., 1989; Aha & Kibler, 1988) y transforma el problema multiclase original (5 niveles de severidad) en una tarea de **clasificación binaria**, que es más robusta dado el tamaño modesto del dataset (303 observaciones).

**Variables de entrada (features) que usaríamos:**

Las 13 columnas restantes: `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`, `oldpeak`, `slope`, `ca`, `thal`.

Antes del entrenamiento sería necesario:
- Tratar los 6 valores faltantes en `ca` y `thal` (imputación o eliminación).
- Aplicar **codificación one-hot** a las variables categóricas nominales (`cp`, `restecg`, `thal`) para evitar imponer un orden artificial.
- Considerar **estandarización** (z-score) o **normalización** (min-max) para las variables continuas si se utilizan algoritmos sensibles a la escala (e.g., SVM, KNN).

**Justificación — ¿Por qué clasificación y no otro tipo de modelo?**

| Tipo de modelo | ¿Aplica? | Razonamiento |
|---|---|---|
| **Clasificación** ✅ | **Sí** | Existe una variable target categórica binaria (enfermedad sí/no). El objetivo es asignar cada paciente a una de dos clases discretas en función de sus características clínicas. |
| Regresión | No | La variable target no es continua. Predecir un valor numérico (e.g., "grado 2.7 de enfermedad") no tiene interpretación clínica útil. |
| Clustering | No | Existe un target etiquetado. Aplicar clustering ignoraría la información supervisada disponible. Podría ser útil como análisis complementario, pero no como enfoque principal. |
| Forecasting | No | No hay dimensión temporal ni serie de tiempo. Los datos son observaciones transversales (*cross-sectional*) de pacientes independientes. |

**Algoritmos candidatos:**

Para un dataset tabular de esta escala (~300 filas, ~13 features), los algoritmos más apropiados según la literatura y los resultados empíricos en benchmarks de clasificación tabular (Grinsztajn et al., 2022) serían:

1. **Regresión Logística:** Baseline interpretable. Ideal para un primer modelo por su transparencia y la posibilidad de interpretar coeficientes como odds ratios, lo que tiene valor clínico directo.
2. **Random Forest / Gradient Boosting (XGBoost, LightGBM):** Modelos ensemble basados en árboles que suelen ofrecer el mejor rendimiento predictivo en datos tabulares de tamaño moderado.
3. **Support Vector Machine (SVM):** Con kernel RBF, puede capturar fronteras de decisión no lineales.

> **Referencias:**
> - Detrano, R. et al. (1989). International application of a new probability algorithm for the diagnosis of coronary artery disease. *American Journal of Cardiology*, 64(5), 304–310.
> - Grinsztajn, L., Oyallon, E., & Varoquaux, G. (2022). Why do tree-based models still outperform deep learning on tabular data? *NeurIPS 2022 Datasets and Benchmarks Track*.


# Etapa 5 — Uso previsto

Por último, pongan el modelo en contexto. Un modelo no se construye en el vacío: sirve para **ayudar a tomar una decisión**.


### ✏️ Su respuesta — Etapa 5

**¿Para qué se usaría el modelo?**

El modelo se utilizaría como **herramienta de apoyo al diagnóstico temprano (*Clinical Decision Support System*, CDSS)** para la detección de enfermedad coronaria. Concretamente, a partir de variables clínicas de rutina (edad, presión arterial, colesterol, resultados de ECG y prueba de esfuerzo), el modelo generaría una **probabilidad estimada** de que el paciente padezca enfermedad cardíaca.

Este tipo de sistema no reemplaza el juicio clínico ni la angiografía diagnóstica, sino que actúa como un **filtro de priorización (*triage*)**: permite identificar a los pacientes de alto riesgo que deberían ser derivados a estudios invasivos de forma prioritaria, y a los de bajo riesgo que pueden ser monitoreados de manera conservadora.

**¿Quién lo usaría?**

- **Médicos de atención primaria y clínicos generales:** que son el primer punto de contacto del paciente y necesitan decidir si derivar a cardiología.
- **Cardiólogos en servicios de urgencias:** para triaje rápido en pacientes que presentan síntomas inespecíficos.
- **Sistemas de salud pública:** como herramienta de *screening* poblacional en campañas de prevención cardiovascular, especialmente en regiones con acceso limitado a especialistas.

**¿Qué decisión ayudaría a tomar?**

Un ejemplo concreto: un paciente de 55 años, masculino, con dolor de pecho no anginoso, presión arterial de 140 mmHg y colesterol de 260 mg/dl consulta en una guardia rural sin cardiólogo disponible. El modelo arroja una probabilidad del 78% de enfermedad cardíaca. Esta predicción fundamenta la decisión de **derivación urgente** a un centro con servicio de hemodinamia para realizar una angiografía coronaria, en lugar de un seguimiento ambulatorio que podría retrasar un diagnóstico crítico.

Según la OMS (2021), la detección temprana y la intervención oportuna pueden reducir la mortalidad cardiovascular en hasta un 30%, lo que otorga al modelo un potencial de impacto significativo.

**Riesgos y consideraciones éticas:**

1. **Falsos negativos (el riesgo más grave).** Un paciente con enfermedad clasificado como sano podría no recibir tratamiento oportuno, con consecuencias potencialmente fatales (infarto de miocardio, muerte súbita). En aplicaciones médicas, es preferible optimizar el **recall** (sensibilidad) sobre la precisión: es clínicamente más aceptable derivar a un paciente sano de más (falso positivo) que dejar sin diagnóstico a un enfermo (falso negativo). La elección del umbral de decisión del clasificador (*threshold*) debe reflejar esta asimetría de costos (Metz, 1978).

2. **Sesgo de representación en el dataset.**
   - *Sesgo de sexo:* El dataset contiene aproximadamente el doble de hombres que de mujeres. Los síntomas de enfermedad coronaria en mujeres suelen ser atípicos (dolor epigástrico, disnea sin dolor precordial), lo cual está subrepresentado en el dataset. Un modelo entrenado con esta distribución podría mostrar menor rendimiento predictivo para mujeres (Puyol-Antón et al., 2021).
   - *Sesgo geográfico y temporal:* Los datos provienen exclusivamente de la Cleveland Clinic en la década de 1980. Las características demográficas, hábitos alimentarios y protocolos clínicos de esa época y región pueden no ser generalizables a poblaciones actuales de otras latitudes.

3. **Datos sensibles y privacidad.** Los datos clínicos son información de salud protegida (*Protected Health Information*, PHI) bajo regulaciones como HIPAA (EE.UU.), GDPR (UE) y la Ley 25.326 de Protección de Datos Personales (Argentina). La implementación en producción requeriría anonimización, consentimiento informado y auditoría de acceso.

4. **Riesgo de automatización excesiva (*automation bias*).** Si los clínicos depositan confianza ciega en el modelo, podrían omitir su propio juicio clínico. Todo CDSS debe presentarse explícitamente como **herramienta de apoyo**, no como sustituto del diagnóstico profesional (Cabitza et al., 2017).

> **Referencias:**
> - OMS (2021). Enfermedades cardiovasculares. https://www.who.int/es/news-room/fact-sheets/detail/cardiovascular-diseases-(cvds)
> - Metz, C. E. (1978). Basic principles of ROC analysis. *Seminars in Nuclear Medicine*, 8(4), 283–298.
> - Cabitza, F., Rasoini, R., & Gensini, G. F. (2017). Unintended consequences of machine learning in medicine. *JAMA*, 318(6), 517–518.
> - Puyol-Antón, E. et al. (2021). Fairness in cardiac MR image analysis. *MICCAI*, 12903, 413–423.


# Cierre y entrega

Antes de entregar, verifiquen que:

- [ ] Eligieron un dataset y lo cargaron correctamente (Etapa 1).
- [ ] Identificaron y clasificaron los tipos de datos (Etapa 2).
- [ ] Ejecutaron el EDA con las herramientas adecuadas al tipo de datos (Etapa 3).
- [ ] Propusieron y justificaron un tipo de modelo (Etapa 4).
- [ ] Explicaron el uso previsto y sus consideraciones (Etapa 5).
- [ ] **Todas las celdas están ejecutadas** y sus salidas son visibles.

**Para entregar:** `Archivo → Descargar → Descargar .ipynb`, o compartan el enlace de Colab con permiso de lectura.

---

*Fundamentos de Aprendizaje Automático · 2026 · Universidad Austral*
